# Feature Engineering

## Financial Fraud Detection System

### Objective

Create meaningful features that improve fraud detection performance while maintaining interpretability.

The features are designed based on observations from Exploratory Data Analysis and domain knowledge in financial fraud detection.

In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

In [3]:
transaction = pd.read_csv("../data/raw/train_transaction.csv")
identity = pd.read_csv("../data/raw/train_identity.csv")

train = transaction.merge(
    identity,
    on="TransactionID",
    how="left"
)

In [4]:
print("=" * 60)
print("Dataset Shape")
print("=" * 60)

print(f"Rows    : {train.shape[0]:,}")
print(f"Columns : {train.shape[1]}")

train.head()

Dataset Shape
Rows    : 590,540
Columns : 434


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [5]:
missing_percentage = (train.isnull().mean() * 100).sort_values(ascending=False)

missing_df = pd.DataFrame({
    "Feature": missing_percentage.index,
    "Missing %": missing_percentage.values
})

missing_df.head(20)

,Feature,Missing %
0,id_24,99.196159
1,id_25,99.130965
2,id_07,99.127070
3,id_08,99.127070
4,id_21,99.126393
5,id_26,99.125715
6,id_27,99.124699
7,id_23,99.124699
8,id_22,99.124699
9,dist2,93.628374


In [6]:
drop_cols = missing_percentage[missing_percentage > 90].index

print(f"Columns to Drop (>90% missing): {len(drop_cols)}")

drop_cols

Columns to Drop (>90% missing): 12


Index(['id_24', 'id_25', 'id_07', 'id_08', 'id_21', 'id_26', 'id_27', 'id_23',
       'id_22', 'dist2', 'D7', 'id_18'],
      dtype='str')

In [7]:
high_value_threshold = train["TransactionAmt"].quantile(0.95)

train["HighValueFlag"] = (
    train["TransactionAmt"] > high_value_threshold
).astype(int)

train[["TransactionAmt", "HighValueFlag"]].head()

,TransactionAmt,HighValueFlag
0,68.5,0
1,29.0,0
2,59.0,0
3,50.0,0
4,50.0,0


In [8]:
train["LogTransactionAmt"] = np.log1p(train["TransactionAmt"])

train[["TransactionAmt", "LogTransactionAmt"]].head()

,TransactionAmt,LogTransactionAmt
0,68.5,4.241327
1,29.0,3.401197
2,59.0,4.094345
3,50.0,3.931826
4,50.0,3.931826


In [9]:
train["AmountPercentile"] = train["TransactionAmt"].rank(pct=True)

train[["TransactionAmt", "AmountPercentile"]].head()

,TransactionAmt,AmountPercentile
0,68.5,0.499216
1,29.0,0.122691
2,59.0,0.450009
3,50.0,0.331933
4,50.0,0.331933


In [10]:
train["HasEmail"] = train["P_emaildomain"].notnull().astype(int)

train[["P_emaildomain", "HasEmail"]].head()

,P_emaildomain,HasEmail
0,NaN,0
1,gmail.com,1
2,outlook.com,1
3,yahoo.com,1
4,gmail.com,1


In [11]:
train["HasDevice"] = train["DeviceType"].notnull().astype(int)

train[["DeviceType", "HasDevice"]].head()

,DeviceType,HasDevice
0,NaN,0
1,NaN,0
2,NaN,0
3,NaN,0
4,mobile,1


In [12]:
identity_cols = [col for col in train.columns if col.startswith("id_")]

train["IdentityInfoCount"] = train[identity_cols].notnull().sum(axis=1)

train[["IdentityInfoCount"]].head()

,IdentityInfoCount
0,0
1,0
2,0
3,0
4,21


In [13]:
train["AmountCategory"] = pd.qcut(
    train["TransactionAmt"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

train["AmountCategory"].value_counts()

AmountCategory
High         149442
Low          147639
Medium       147631
Very High    145828
Name: count, dtype: int64

In [14]:
train.to_csv(
    "../data/processed/train_engineered.csv",
    index=False
)

print("Engineered dataset saved successfully!")

Engineered dataset saved successfully!


## Engineered Features Summary

| Feature | Purpose |
|----------|---------|
| HighValueFlag | Detect unusually large transactions |
| LogTransactionAmt | Reduce skewness |
| AmountPercentile | Relative transaction size |
| HasEmail | Indicates email availability |
| HasDevice | Indicates device information availability |
| IdentityInfoCount | Counts available identity attributes |
| AmountCategory | Categorizes transaction amount into quartiles |

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/train_engineered.csv")

demo = df.sample(
    n=min(300, len(df)),
    random_state=42
)

demo.to_csv(
    "../data/processed/demo.csv",
    index=False
)

print("Demo created:", demo.shape)

Demo created: (300, 441)
